In [3]:
import requests

# Download a small part of Alice in Wonderland to use as training data
url = "https://www.gutenberg.org/files/11/11-0.txt"
text = requests.get(url).text
start = text.find("CHAPTER I")
# Save the first 20,000 characters to data.txt
with open('data.txt', 'w', encoding='utf-8') as f:
    f.write(text[start:start+20000])

print("data.txt created successfully.")

data.txt created successfully.


In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN, Activation
from tensorflow.keras.optimizers import RMSprop
import random

# ==========================================
# Task 1: Load Text
# ==========================================
filename = 'data.txt'
# Reads the file directly. Ensure data.txt is in the same folder.
text = open(filename, 'r', encoding='utf-8').read().lower()
print('Corpus length:', len(text))

# ==========================================
# Task 2: Vectorization
# ==========================================
chars = sorted(list(set(text)))
print('Total chars:', len(chars))
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

# Cut the text in semi-redundant sequences of maxlen characters
maxlen = 40
step = 3
sentences = []
next_chars = []

for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i + maxlen])
print('Nb sequences:', len(sentences))

print('Vectorization...')
x = np.zeros((len(sentences), maxlen, len(chars)), dtype=bool)
y = np.zeros((len(sentences), len(chars)), dtype=bool)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1

# ==========================================
# Task 4 & 5: Build and Train Models
# ==========================================

def build_model(model_type):
    model = Sequential()
    if model_type == 'rnn':
        model.add(SimpleRNN(128, input_shape=(maxlen, len(chars))))
    else:
        model.add(LSTM(128, input_shape=(maxlen, len(chars))))

    model.add(Dense(len(chars)))
    model.add(Activation('softmax'))

    optimizer = RMSprop(learning_rate=0.01)
    model.compile(loss='categorical_crossentropy', optimizer=optimizer)
    return model

# Train RNN
print("\n--- Training Simple RNN ---")
model_rnn = build_model('rnn')
model_rnn.fit(x, y, batch_size=128, epochs=5)

# Train LSTM
print("\n--- Training LSTM ---")
model_lstm = build_model('lstm')
model_lstm.fit(x, y, batch_size=128, epochs=5)

# ==========================================
# Task 6 & 7: Generate Text
# ==========================================
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-7) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(model, start_string):
    generated = start_string
    sentence = start_string
    print('Generating with seed: "' + start_string + '"')

    for i in range(400):
        x_pred = np.zeros((1, maxlen, len(chars)))
        for t, char in enumerate(sentence):
            if char in char_indices:
                x_pred[0, t, char_indices[char]] = 1

        preds = model.predict(x_pred, verbose=0)[0]
        next_index = sample(preds, 0.5)
        next_char = indices_char[next_index]

        generated += next_char
        sentence = sentence[1:] + next_char
    return generated

# Select a random seed sequence
start_index = random.randint(0, len(text) - maxlen - 1)
seed_text = text[start_index: start_index + maxlen]

print("\n--- RNN Result ---")
print(generate_text(model_rnn, seed_text))

print("\n--- LSTM Result ---")
print(generate_text(model_lstm, seed_text))

Corpus length: 20000
Total chars: 44
Nb sequences: 6654
Vectorization...

--- Training Simple RNN ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 3.8875
Epoch 2/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 3.3123
Epoch 3/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 2.9155
Epoch 4/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 2.7635
Epoch 5/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - loss: 2.6283

--- Training LSTM ---
Epoch 1/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 7s 98ms/step - loss: 3.5079
Epoch 2/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 105ms/step - loss: 2.9499
Epoch 3/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 119ms/step - loss: 2.6684
Epoch 4/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 98ms/step - loss: 2.5034
Epoch 5/5
52/52 ━━━━━━━━━━━━━━━━━━━━ 7s 126ms/step - loss: 2.3732

--- RNN Result ---
Generating with seed: " at her rather
inquisitively, and seemed"
 at her rather
inquisitively, and seemed ooae aar ale aa  araale aas aathe washeree ine hathe war aaee ae aataaasheas ae ae aa ae ae ar aa tare ae ae aaatheas ar areay she teehe ineae aaae aaae ar aar aala are aae eatheraraehe warieeearee 